# FORTRAN - typy zmiennych
Pierwsze zajęcia: zmienne, typy danych, precyzja i bezpieczne obliczenia.

Cele zajęć:
- rozumieć podstawowe typy danych w Fortranie
- umieć deklarować i inicjalizować zmienne
- wiedzieć, czym jest `kind` i jak kontrolować precyzję
- zauważać typowe błędy początkujących (przepełnienie, mieszanie typów)

Plan spotkania (krok po kroku):
1. Szybki przegląd typów i deklaracji
2. Bezpieczny start programu (`implicit none`)
3. Konwersje typów i pułapki dzielenia
4. Precyzja: `kind`, `huge`, `precision`, `range`
5. Dobór typu przez `selected_int_kind` i `selected_real_kind`
6. Najczęstsze błędy początkujących + ćwiczenia + ściąga

Materiał pomocniczy:
https://www.tutorialspoint.com/fortran/fortran_data_types.htm

## 0) Szybki przegląd typów i deklaracji

Zanim przejdziemy do pułapek i precyzji, zobaczmy krótki program z najczęściej używanymi typami: `integer`, `real`, `complex`, `character`, `logical`.

Pierwszy raz pojawia się tu też `kind`, więc od razu krótko:
- `kind` określa wariant danego typu liczbowego używany przez kompilator.
- W praktyce używamy go wtedy, gdy chcemy przechować większe liczby albo pracować z większą precyzją.
- Zapis `integer(kind=8)` oznacza: „użyj takiego wariantu typu `integer`, który kompilator oznacza identyfikatorem `8`”.

Krótka wersja, którą warto zapamiętać:
- `kind` nie jest liczbą bajtów
- `kind` jest identyfikatorem reprezentacji typu
- zwykle jego wartość pokrywa się z liczbą bajtów, ale nie jest to reguła języka

Co zwykle oznaczają konkretne liczby w `kind`:
- `integer(kind=4)` - na wielu kompilatorach odpowiada 32-bitowemu typowi całkowitemu
- `integer(kind=8)` - na wielu kompilatorach odpowiada 64-bitowemu typowi całkowitemu
- `real(kind=4)` - na wielu kompilatorach odpowiada pojedynczej precyzji
- `real(kind=8)` - na wielu kompilatorach odpowiada podwójnej precyzji

Ważne:
- te liczby są zależne od kompilatora i platformy
- `kind=8` nie znaczy automatycznie „8 bajtów” ani „64 bity” w każdej sytuacji
- dlatego w bardziej przenośnym kodzie lepiej używać `selected_int_kind(...)` i `selected_real_kind(...)`

Na tym etapie wystarczy intuicja:
- zwykły `integer` -> typ całkowity do typowych zastosowań
- `integer(kind=8)` -> typ całkowity do większych liczb

Dokładniejsze znaczenie `kind` omówimy później, ale już teraz warto zauważyć, że zmiana `kind` zmienia możliwości typu.

In [ ]:
program demo_typy
  implicit none

  integer          :: i = 68312
  integer(kind=8)  :: j = 2147483648_8
  real             :: liczba = 8e-9
  complex          :: liczba_zesp = (2.0, -3.0)
  character(len=5) :: a = 'kot'
  logical          :: warunek = .true.

  write(*,*) 'integer i =', i
  write(*,*) 'integer(kind=8) j =', j
  write(*,*) 'real liczba =', liczba
  write(*,*) 'complex liczba_zesp =', liczba_zesp
  write(*,*) 'character a =', trim(a)
  write(*,*) 'logical warunek =', warunek

end program demo_typy

       68312


## 1) Minimalny program i bezpieczny start

Od początku warto używać `implicit none`, żeby kompilator wymagał jawnych deklaracji wszystkich zmiennych. Dzięki temu literówki i błędy nazw nie przechodzą „po cichu”.

Poniżej: prosty przykład deklaracji i inicjalizacji podstawowych typów.

In [ ]:
program start_bezpiecznie
  implicit none

  integer :: wiek
  real :: wzrost
  character(len=20) :: imie
  logical :: student

  wiek = 20
  wzrost = 1.82
  imie = 'Ala'
  student = .true.

  write(*,*) 'Imię:', trim(imie)
  write(*,*) 'Wiek:', wiek
  write(*,*) 'Wzrost [m]:', wzrost
  write(*,*) 'Czy student?:', student

end program start_bezpiecznie

## Komentarze w kodzie: po co i jak je pisać

Komentarze są ważne, bo pomagają szybko zrozumieć intencję programu:
- co robi dany fragment kodu
- dlaczego coś zostało zapisane w taki sposób
- jakie są założenia albo ograniczenia

W Fortranie komentarz jednoliniowy zaczynamy od znaku `!`. Wszystko po `!` w danej linii jest traktowane jako komentarz.

Dobra praktyka: komentuj sens i zamiar kodu, a nie oczywiste szczegóły.

In [ ]:
program komentarze_demo
  implicit none

  integer :: a, b
  real :: wynik

  ! Ustawiamy dane wejściowe do przykładu
  a = 7
  b = 2

  ! Zamieniamy liczby na real, żeby uniknąć dzielenia całkowitego
  wynik = real(a) / real(b)

  write(*,*) 'Wynik =', wynik  ! Wypisujemy wynik obliczeń

end program komentarze_demo

### Mini-checkpoint 1

1. Dlaczego `implicit none` warto mieć w każdym programie?
2. Jaka będzie wartość `wynik`, jeśli `a=9`, `b=4` i napiszesz `wynik = a / b`?
3. Co może pójść nie tak, gdy próbujesz zapisać bardzo dużą liczbę całkowitą?

<details>
<summary>Odpowiedzi dla prowadzącego</summary>

- `implicit none` wymusza deklaracje zmiennych i pomaga wykrywać literówki na etapie kompilacji.
- Dla `a=9`, `b=4`: `a / b` wykona dzielenie całkowite, więc wynik to `2` (a po przypisaniu do `real` będzie `2.0`).
- Może wystąpić przepełnienie typu całkowitego i niepoprawny wynik; dlatego później wprowadzamy temat doboru precyzji (`kind`).

</details>

## 2) Operacje na typach i konwersje

Fortran pozwala mieszać typy, ale warto robić to świadomie. Najczęściej używa się funkcji `real(...)` i `int(...)`.

Kluczowa pułapka: jeśli `a` i `b` są typu `integer`, to `a / b` wykona dzielenie całkowite, nawet gdy wynik przypisujesz do `real`.

Poniższy kod pokazuje różnicę między dzieleniem całkowitym a rzeczywistym.

Dobrą praktyką jest wypisywanie obu wariantów obok siebie, żeby szybko wychwycić, czy nie zgubiliśmy części ułamkowej.

In [ ]:
program konwersje_typow
  implicit none

  integer :: a, b
  real :: iloraz_calkowity, iloraz_realny

  a = 7
  b = 2

  iloraz_calkowity = a / b          ! najpierw dzielenie całkowite: 3
  iloraz_realny = real(a) / real(b) ! dzielenie rzeczywiste: 3.5

  write(*,*) 'a/b jako integer->real:', iloraz_calkowity
  write(*,*) 'a/b jako real:', iloraz_realny

end program konwersje_typow

## 3) Precyzja i zakres typu

Po konwersjach przechodzimy do pytania: „jak duże i jak dokładne liczby mogę bezpiecznie przechowywać?”.
Poniższy przykład pokazuje, jak odczytać informacje o typie przez `kind`, `huge`, `precision` i `range`.

Jak rozumieć `kind` w praktyce:
- `kind(x)` zwraca identyfikator wariantu typu użytego przez kompilator.
- To **nie musi** być liczba bajtów (choć czasem bywa z nią powiązana).
- Dwie zmienne tego samego typu mogą mieć różne `kind`, np. `real` i `real(kind=8)`.
- `kind` służy do świadomego wyboru precyzji i zakresu liczb.

In [3]:
program demo

integer          :: i=68312
integer (kind=8) :: j=2147483648_8
real             :: liczba=8e-9
real    (kind=8) :: druga_liczba=123.456

write(*,*)i, kind(i), huge(i), range(i)

end program demo

       68312           4  2147483647           9


### Skąd bierze się liczba bitów i zakres wartości?

Komputer zapisuje liczby w systemie binarnym, czyli za pomocą bitów `0` i `1`. Każdy kolejny bit podwaja liczbę możliwych kombinacji.

Jeśli mamy `n` bitów, to można zapisać dokładnie $2^n$ różnych stanów.
Przykłady:
- 1 bit -> $2^1 = 2$ stany
- 2 bity -> $2^2 = 4$ stany
- 8 bitów -> $2^8 = 256$ stanów

Dla liczb bez znaku zakres wynosi od $0$ do $2^n - 1$.
Przykład dla 8 bitów: od $0$ do $255$.

Dla liczb ze znakiem jeden bit jest zwykle wykorzystywany na informację o znaku, więc zakres dodatni i ujemny jest mniejszy.
W praktyce dla liczb całkowitych zapisanych na 32 bitach typowy zakres to około od $-2^{31}$ do $2^{31}-1$.

To właśnie dlatego większa liczba bitów pozwala zapisywać większe liczby:
- więcej bitów -> więcej możliwych kombinacji
- więcej kombinacji -> większy zakres wartości

W Fortranie nie operujemy zwykle bezpośrednio na liczbie bitów typu, tylko wybieramy odpowiedni wariant reprezentacji przez `kind` albo przez `selected_*_kind`.

## 4) Automatyczny dobór typu: `selected_*_kind`

Gdy wymagania rosną, lepiej nie zgadywać numeru `kind`, tylko dobrać go funkcjami `selected_int_kind(...)` i `selected_real_kind(...)`.
To podejście jest bardziej przenośne między kompilatorami i platformami.

Jakie argumenty przyjmują te funkcje:
- `selected_int_kind(R)`
  - `R` oznacza wymaganą liczbę cyfr dziesiętnych dla typu całkowitego
  - intuicyjnie: chcesz móc bezpiecznie zapisać liczby rzędu około $10^R$
- `selected_real_kind(P, R)`
  - `P` oznacza wymaganą precyzję, czyli liczbę cyfr znaczących
  - `R` oznacza wymagany zakres dziesiętny

Przykłady:
- `selected_int_kind(9)` -> typ całkowity mieszczący liczby mniej więcej do 9 cyfr
- `selected_real_kind(15, 300)` -> typ rzeczywisty o około 15 cyfrach precyzji i bardzo dużym zakresie

Materiał pomocniczy:
https://www.tutorialspoint.com/fortran/fortran_numeric_precision.htm

In [ ]:
program whatkind
  implicit none

  integer, parameter :: ki = selected_int_kind(12)
  integer(kind=ki) :: i
  integer(kind=8)  :: j

  i = 2147483647
  j = 2147483648_8

  write(*,*) 'i =', i, ' kind(i)=', kind(i), ' huge(i)=', huge(i), ' range(i)=', range(i)
  write(*,*) 'j =', j, ' kind(j)=', kind(j), ' huge(j)=', huge(j), ' range(j)=', range(j)

end program whatkind

### Mini-checkpoint 2

1. Czym różni się `kind(...)` od `range(...)`?
2. Jak sprawdzić maksymalną wartość dla zmiennej całkowitej?
3. Po co używamy `selected_int_kind(...)` i `selected_real_kind(...)`?

<details>
<summary>Odpowiedzi dla prowadzącego</summary>

- `kind(...)` opisuje wariant implementacji typu (precyzję/rozmiar), a `range(...)` podaje przybliżony zakres dziesiętny wartości.
- Używamy `huge(x)`, np. `huge(i)` dla zmiennej `integer :: i`.
- Funkcje `selected_*_kind` pozwalają dobrać typ na podstawie wymaganej precyzji i zakresu, zamiast zgadywać numer `kind` ręcznie.

</details>

## 5) Najczęstsze błędy początkujących (zły vs dobry kod)

Po omówieniu typów, konwersji i precyzji warto zebrać najczęstsze błędy w jednym miejscu.

### 1) Brak `implicit none`
Zły kod (literówka przechodzi niezauważona):
```fortran
program zly_1
  integer :: liczba
  liczab = 10
  write(*,*) liczba
end program zly_1
```

Dobry kod (kompilator od razu łapie błąd):
```fortran
program dobry_1
  implicit none
  integer :: liczba
  liczba = 10
  write(*,*) liczba
end program dobry_1
```

### 2) Niespodziewane dzielenie całkowite
Zły kod:
```fortran
integer :: a, b
real :: wynik
a = 7
b = 2
wynik = a / b
```

Dobry kod:
```fortran
integer :: a, b
real :: wynik
a = 7
b = 2
wynik = real(a) / real(b)
```

### 3) Przepełnienie typu `integer`
Zły kod:
```fortran
integer :: x
x = 2147483648
```

Dobry kod:
```fortran
integer(kind=8) :: x
x = 2147483648_8
```

## 6) Ćwiczenia na pierwsze zajęcia

Teraz przejdź do krótkiej praktyki, żeby utrwalić materiał z całego bloku.

1. Zmień w przykładzie wartości `a` i `b` i sprawdź, kiedy wynik dzielenia całkowitego różni się od rzeczywistego.
2. Dodaj zmienną `real(kind=8)` i wypisz jej `kind`, `precision` oraz `range`.
3. Sprawdź, co się stanie po próbie przypisania bardzo dużej liczby do zwykłego `integer`.
4. Dla chętnych: użyj `selected_int_kind` oraz `selected_real_kind` do zdefiniowania parametrów typu.

In [ ]:
program cwiczenie_ucznia
  implicit none

  integer :: a, b
  real :: wynik

  a = 15
  b = 4

  ! TODO 1: wypisz a i b
  ! TODO 2: policz i wypisz dzielenie całkowite
  ! TODO 3: policz i wypisz dzielenie rzeczywiste (z real(...))

  wynik = real(a) / real(b)
  write(*,*) 'wynik rzeczywisty =', wynik

end program cwiczenie_ucznia

### Mini-checkpoint 3

1. Podaj przykład sytuacji, w której jawna konwersja typu (`real(...)`, `int(...)`) zapobiega błędowi.
2. Jakim jednym nawykiem najbardziej poprawisz jakość kodu od pierwszego dnia?
3. Jak dobrać typ liczbowy, gdy wiesz, że wartości mogą być bardzo duże?

<details>
<summary>Odpowiedzi dla prowadzącego</summary>

- Np. przy dzieleniu `a/b`: zapis `real(a)/real(b)` zapobiega niechcianemu dzieleniu całkowitemu.
- Najlepszy nawyk: zawsze zaczynać program od `implicit none`.
- Dla bardzo dużych liczb użyć większego `kind` (np. `integer(kind=8)`) lub dobrać go przez `selected_int_kind(...)` / `selected_real_kind(...)`.

</details>

## 7) Ściąga (1 strona) - podsumowanie pierwszych zajęć

Poniżej wersja skrócona do szybkiej powtórki (lub wydruku).

### 1) Typy i deklaracje
| Typ | Przykład deklaracji | Zastosowanie |
|---|---|---|
| `integer` | `integer :: i` | liczby całkowite |
| `real` | `real :: x` | liczby rzeczywiste |
| `complex` | `complex :: z` | liczby zespolone |
| `logical` | `logical :: ok` | wartości logiczne |
| `character` | `character(len=20) :: imie` | napisy |

### 2) Konwersje i pułapki
| Temat | Zły zapis | Dobry zapis |
|---|---|---|
| Dzielenie | `wynik = a / b` | `wynik = real(a) / real(b)` |
| Duże liczby całkowite | `integer :: x; x = 2147483648` | `integer(kind=8) :: x; x = 2147483648_8` |
| Konwersja `real -> integer` | `int(r)` bez refleksji | `int(r)` świadomie (obcina część ułamkową) |

### 3) Funkcje diagnostyczne
| Funkcja | Co zwraca |
|---|---|
| `kind(x)` | identyfikator wariantu typu użytego przez kompilator |
| `huge(x)` | największą reprezentowalną wartość |
| `precision(x)` | liczbę cyfr znaczących (`real`, `complex`) |
| `range(x)` | przybliżony zakres dziesiętny |

Krótka wersja o `kind`:
- `kind` nie jest liczbą bajtów
- `kind` jest identyfikatorem reprezentacji typu
- czasem jego wartość pokrywa się z liczbą bajtów, ale nie jest to reguła języka

Uwaga o `kind`:
- `kind` to identyfikator typu, a nie gwarantowana liczba bajtów ani bitów.
- Na wielu kompilatorach `integer(kind=4)` odpowiada typowi 32-bitowemu, a `integer(kind=8)` typowi 64-bitowemu.
- Dla `real`: `kind=4` często oznacza pojedynczą precyzję, a `kind=8` podwójną.
- To są typowe konwencje, ale zależą od kompilatora.
- Przykład: `kind(1.0)` i `kind(1.0d0)` zwykle dają różne wartości (pojedyncza vs podwójna precyzja).

### 4) Dobór typu przez `selected_*_kind`
- `selected_int_kind(R)`
  - `R`: wymagana liczba cyfr dziesiętnych dla typu całkowitego
- `selected_real_kind(P, R)`
  - `P`: wymagana liczba cyfr znaczących
  - `R`: wymagany zakres dziesiętny

### 5) Szybki szablon startowy
```fortran
program start
  implicit none
  integer :: a, b
  real :: wynik

  a = 7
  b = 2
  wynik = real(a) / real(b)

  write(*,*) 'Wynik =', wynik
end program start
```

### 6) Stałe i parametry
- Używaj `parameter`, gdy wartość ma być stała w całym programie.
- Przykład: `integer, parameter :: n = 10`

```fortran
integer, parameter :: n = 10
real :: tab(n)
```

### 7) Podstawy debugowania: ostrzeżenia kompilatora i flagi kompilacji
- Na etapie nauki kompiluj z ostrzeżeniami i kontrolą błędów wykonania.
- Przykładowa komenda dla `gfortran`:

```bash
gfortran -Wall -Wextra -Wconversion -fcheck=all -g -O0 program.f90 -o program
```

Skrót najważniejszych flag:
- `-Wall -Wextra` -> dodatkowe ostrzeżenia
- `-Wconversion` -> ostrzeżenia o niejawnych konwersjach
- `-fcheck=all` -> sprawdzanie błędów w czasie wykonania
- `-g` -> informacje debugowe (np. do `gdb`)
- `-O0` -> brak optymalizacji, łatwiejsze debugowanie